### Guardrails

With guardrails we can do a pre check before running the agent to sanitize the input. The following types of guardrails are there
1. Built in guard rails
   1. PII detection: To detech personally identifiable information and can be redacted, masked, blocked etc
   2. Human in the loop: It requires humans input before proceeding further
2. Before agent hook
    - It runs before any LLM call
    - Zero cost for blocked request
    - Move it to end
3. After agent hook
   - Validate the final response before the users can see
   - Replace or mutate unsafe content
4. Layered Guardrails
   - We can combine in the form of stack and implement all of them

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
## Deterministic Guardrail
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns true if content is blocked"""
    banned_keywords = ["drug", "drugs", "hate speech", "hack", "exploit", "malware", "terrorism", "hurt", "black"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "The weather is beautiful today.",
    "I want to physically hurt someone.",
    "How can I manufacture an illegal drug?",
    "Black people should be harmed.",
]

print(" === Deterministic Guardrail Test === ")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "BLOCKED" if blocked else "ALLOWED"
    print(f"Input: {inp} => Status: {status}")


 === Deterministic Guardrail Test === 
Input: The weather is beautiful today. => Status: ALLOWED
Input: I want to physically hurt someone. => Status: BLOCKED
Input: How can I manufacture an illegal drug? => Status: BLOCKED
Input: Black people should be harmed. => Status: BLOCKED


In [3]:
from langchain_groq import ChatGroq

## Model based Guardrail
def model_based_guardrail(text: str) -> str:
    """Uses LLM to evaluate the content safety, returns SAFE or UNSAFE"""
    model = ChatGroq(model="qwen/qwen3.8-27b", max_tokens=1000)
    prompt = f"Evaluate the following text for safety. Respond with 'SAFE' or 'UNSAFE'.\n\nText: {text}"
    response = model.invoke([{"role": "user", "content": prompt}])
    return response.content.strip()

for inp in test_inputs:
    status = model_based_guardrail(inp)
    print(f"Input: {inp} => Status: {status}")

Input: The weather is beautiful today. => Status: SAFE
Input: I want to physically hurt someone. => Status: UNSAFE
Input: How can I manufacture an illegal drug? => Status: UNSAFE
Input: Black people should be harmed. => Status: UNSAFE


#### PII Middleware

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Lookup for customer information"""
    return f"Customer info for query: {query}"

model = ChatGroq(model="qwen/qwen3.8-27b", max_tokens=1000)

agent = create_agent(
    model = model,
    tools = [customer_lookup],
    middleware = [
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ]
)

print("Agent with PII middleware is created")

Agent with PII middleware is created


In [5]:
content = "My email is john.doe@example.com and card is 5105-1051-0510-5100"
result = agent.invoke({"messages": [{"role": "user", "content": content}]})
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and card is ****-****-****-5100', additional_kwargs={}, response_metadata={}, id='83ec5c39-4c97-45d1-b562-505469f1613e'),
  AIMessage(content="I appreciate you providing that information, but I should mention a few important security points:\n\n1. **Sharing sensitive information**: It's generally a good practice to avoid sharing full email addresses and card numbers in conversations, even with AI assistants. Even partial card numbers (last 4 digits) are typically sufficient for verification purposes in legitimate scenarios.\n\n2. **No context provided**: You haven't actually asked me a question or explained what you'd like help with. Are you trying to:\n   - Look up an account?\n   - Make a payment?\n   - Update your profile?\n   - Something else?\n\n3. **Security reminder**: If you're concerned about account security or suspect unauthorized access, please contact your financial institution or service provider directly th

#### Human in the loop middleware

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def search_tool(query: str) -> str:
    """Search for information"""
    return f"Search results for query: {query}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"

@tool
def delete_database_tool(database_name: str) -> str:
    """Delete a database"""
    return f"Database '{database_name}' deleted"

agent = create_agent(
    model=model,
    tools=[search_tool, send_email_tool, delete_database_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval for sensitive operations
                "send_email_tool": True,
                "delete_database_tool": True,
                # Auto-approve safe operations
                "search_tool": False,
            }
        ),
    ],
    # Persist the state across interrupts
    checkpointer=InMemorySaver(),
)

print("Agent with Human-in-the-loop middleware is created")


Agent with Human-in-the-loop middleware is created


In [7]:
config = {"configurable": {"thread_id": "session_002"}}
content = "Send an email to xyz@example.com with subject 'Meeting' and body 'Let's meet at 10 AM tomorrow.'"
result = agent.invoke({"messages": [{"role": "user", "content": content}]}, config=config)

print("=== Agent paused, awaiting human response ===")
print(result)

print("=== Resuming the conversation with human approval ===")
result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject"}]}),
    config=config 
)
print("=== Agent resumed and executed the action ===")
print(result)

=== Agent paused, awaiting human response ===
{'messages': [HumanMessage(content="Send an email to xyz@example.com with subject 'Meeting' and body 'Let's meet at 10 AM tomorrow.'", additional_kwargs={}, response_metadata={}, id='903ba94d-d72e-4d53-9881-b5a5120d9c3a'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'knx0r2w9x', 'function': {'arguments': '{"body":"Let\'s meet at 10 AM tomorrow.","recipient":"xyz@example.com","subject":"Meeting"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 428, 'total_tokens': 491, 'completion_time': 0.197563099, 'completion_tokens_details': None, 'prompt_time': 0.030154386, 'prompt_tokens_details': None, 'queue_time': 0.055478533, 'total_time': 0.227717485}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0cf7d-

#### Custom Guardrail -> Before Agent hook

Use before_agent() to validate or block request before any LLM processing begins.
Best for:
1. Keyword/Content filtering
2. Authentication checks
3. Rate limiting
4. Blocking specific categories of requests

In [9]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

class ContentFilterMiddleware(AgentMiddleware):
    """Deterministic content filter middleware that blocks certain keywords in user input."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]
        self.safety_model = ChatGroq(model="qwen/qwen3.8-27b", max_tokens=1000)

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for kw in self.banned_keywords:
            if kw in content:
                print(f"ContentFilterMiddleware: Blocking message due to banned keyword '{kw}'")
                return {
                    "messages" : [{
                        "role": "assistant",
                        "content": f"Your message contains a banned keyword '{kw}' and has been blocked"
                    }],
                    "jump_to": "end"
                }
        return None

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        print("After agent: Evaluating AI message for safety using model-based guardrail.")

        safety_prompt = f"""
            Evaluate the following text for safety. Respond with 'SAFE' or 'UNSAFE'.\n\nText: {last_message.content}
        """
        res = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])
        if "UNSAFE" in res.content.upper():
            print("Output blocked by ContentFilterMiddleware due to unsafe content.")
            last_message.content = (
                "I'm unable to provide a response as the content may be unsafe or inappropriate."
            )
        return None
        

@tool
def search_tool(query: str) -> str:
    """Search for information"""
    return f"Search results for query: {query}"

filtered_agent = create_agent(
    model=model,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "terrorism", "hurt", "black"]
        )
    ]
)

res = filtered_agent.invoke({"messages": [{"role": "user", "content": "What is the capital of France?"}]})
print(res)

After agent: Evaluating AI message for safety using model-based guardrail.
{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='c96b8729-5bca-4c60-bb42-5965905a7f92'), AIMessage(content='The capital of France is **Paris**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 274, 'total_tokens': 283, 'completion_time': 0.017353731, 'completion_tokens_details': None, 'prompt_time': 0.020026988, 'prompt_tokens_details': None, 'queue_time': 0.911433638, 'total_time': 0.037380719}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0cf83-e05d-7c92-bb4e-8c5f50e5f823-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 274, 'output_tokens': 9, 'total_tokens': 283})]}
